In [ ]:
%%writefile ambiente.h
// Contiene: i parametri, gli ostacoli (ostacoli e fori), la maschera della griglia,
// le sorgenti della passata 1 (bordo degli ostacoli)
#pragma once           // il file viene incluso una sola volta
#include <vector>
#include <cmath>
#include <algorithm>
#include <random>

// Forme con cui si descrive l'ambiente (coordinate nel cubo [-1, 1]^3)
struct Punto   { double x, y, z; };
struct Zona { double x0, x1, y0, y1, z0, z1; };       // parallelepipedo: x0 <= x <= x1, y0 <= y <= y1, z0 <= z <= z1


// AMBIENTE: per cambiarlo si modifica solo questa parte
// =====================================================================
// Stanza = cubo [-1, 1]^3 (20 m di lato), z verticale (z = -1 pavimento). Le 6 pareti sono ostacoli (PARETI).
const double MARGINE = 0.06;                             // distanza minima dagli ostacoli (raggio del drone + sicurezza)
const double D_SICUREZZA = 0.3;                          // distanza dagli ostacoli oltre la quale si va a velocita' piena
const double V_MIN = 0.1;                                // velocita' minima, a ridosso degli ostacoli (> 0)

// Le 6 pareti della stanza sono ostacoli; i muri interni con i fori li crea genera_mappa (sotto)
const double SP = 0.04;                                  // spessore delle pareti della stanza
const std::vector<Zona> PARETI = {                    // le 6 pareti della stanza (pavimento e soffitto compresi)
    { -1.0, -1.0 + SP, -1.0, 1.0, -1.0, 1.0 }, { 1.0 - SP, 1.0, -1.0, 1.0, -1.0, 1.0 },
    { -1.0, 1.0, -1.0, -1.0 + SP, -1.0, 1.0 }, { -1.0, 1.0, 1.0 - SP, 1.0, -1.0, 1.0 },
    { -1.0, 1.0, -1.0, 1.0, -1.0, -1.0 + SP }, { -1.0, 1.0, -1.0, 1.0, 1.0 - SP, 1.0 },
};
// Mappe casuali: K muri perpendicolari a x, ognuno con M fori quadrati di lato casuale.
// A nel primo scomparto, B nell'ultimo. Stesso seed = stessa mappa.
const double SPESSORE = 0.10;                            // spessore dei muri interni
const double X_PRIMO = -0.6, X_ULTIMO = 0.6;             // centro (in x) del primo e dell'ultimo muro
inline Punto A, B;                                       // partenza e arrivo: li sceglie genera_mappa

inline std::vector<Zona> OSTACOLI;                     // ostacoli della mappa attuale (pareti + muri)
inline std::vector<Zona> FORI;                        // zone in cui l'ostacolo viene tolto

inline void genera_mappa(int K, int M, double lato_min, double lato_max, unsigned seed)
{
    std::mt19937 rng(seed);                                                     // generatore di numeri casuali
    auto caso = [&](double a, double b) { return std::uniform_real_distribution<double>(a, b)(rng); };   // numero casuale in [a, b]
    OSTACOLI = PARETI;
    FORI.clear();
    for (int w = 0; w < K; ++w) {
        const double xc = (K == 1) ? 0.0 : X_PRIMO + (X_ULTIMO - X_PRIMO) * w / (K - 1);   // centro del muro w: muri equidistanti
        const double x0 = xc - SPESSORE / 2, x1 = xc + SPESSORE / 2;
        OSTACOLI.push_back({ x0, x1, -1.0, 1.0, -1.0, 1.0 });                    // il muro chiude tutta la sezione della stanza
        for (int f = 0; f < M; ++f) {
            const double lato = caso(lato_min, lato_max);                       // lato del foro
            const double lim = 1.0 - SP - MARGINE - lato / 2;                   // il foro non tocca le pareti della stanza
            const double yc = caso(-lim, lim), zc = caso(-lim, lim);            // centro del foro sul muro
            FORI.push_back({ x0, x1, yc - lato / 2, yc + lato / 2, zc - lato / 2, zc + lato / 2 });
        }
    }
    const double xa = X_PRIMO - SPESSORE / 2, xb = X_ULTIMO + SPESSORE / 2;     // faccia sinistra del primo muro, destra dell'ultimo
    A = { caso(-1.0 + SP + 0.1, xa - 0.1), caso(-0.8, 0.8), caso(-0.8, 0.8) };  // primo scomparto, a piu' di 0.1 da ogni ostacolo
    B = { caso(xb + 0.1, 1.0 - SP - 0.1), caso(-0.8, 0.8), caso(-0.8, 0.8) };   // ultimo scomparto
}
// Livelli di difficolta': piu' muri, meno fori per muro, fori piu' piccoli
struct Livello { int K, M; double lato_min, lato_max; };    // K muri, M fori per muro, lato dei fori in [lato_min, lato_max]
const std::vector<Livello> LIVELLI = {
    { 2, 3, 0.30, 0.50 },                                   // L1: facile
    { 3, 2, 0.25, 0.45 },                                   // L2
    { 4, 2, 0.20, 0.40 },                                   // L3
    { 5, 2, 0.15, 0.35 },                                   // L4
    { 6, 1, 0.15, 0.30 },                                   // L5: difficile, un solo foro per muro
};
// =============================================================================

// Vero se il punto (x, y, z) e' dentro la zona os (bordi compresi)
inline bool dentro(const Zona& os, double x, double y, double z)
{
    return os.x0 <= x && x <= os.x1 && os.y0 <= y && y <= os.y1 && os.z0 <= z && z <= os.z1;
}

// Vero se il punto (x, y, z) e' dentro un ostacolo (i FORI tolgono l'ostacolo)
inline bool ostacolo(double x, double y, double z)
{
    for (const auto& f : FORI)    if (dentro(f, x, y, z)) return false;
    for (const auto& os : OSTACOLI) if (dentro(os, x, y, z)) return true;
    return false;
}

// Maschera della griglia: libero = 1, ostacolo = 0 (resta INF)
inline std::vector<unsigned char> costruisci_maschera(int N)
{
    const double h = 2.0 / (N - 1);
    const int n = N * N * N;
    std::vector<unsigned char> mask(n);
    for (int q = 0; q < n; ++q) {                     // un nodo alla volta
    const int j = q % N, i = (q / N) % N, k = q / (N * N);
    mask[q] = ostacolo(-1.0 + j * h, -1.0 + i * h, -1.0 + k * h) ? 0 : 1;
    }
    // Margine: diventa ostacolo ogni nodo entro MARGINE da un ostacolo
    const int r = (int)std::ceil(MARGINE / h);               // raggio sfera intorno all'ostacolo
    std::vector<unsigned char> margin_mask = mask;
    for (int q = 0; q < n; ++q) {
        if (mask[q]) continue;                                   
    const int k = q / (N * N), i = (q / N) % N, j = q % N;
    const bool bordo = (j > 0 && mask[q - 1]) || (j < N - 1 && mask[q + 1]) ||          // che toccano un nodo libero
                       (i > 0 && mask[q - N]) || (i < N - 1 && mask[q + N]) ||
                       (k > 0 && mask[q - N * N]) || (k < N - 1 && mask[q + N * N]);
    if (!bordo) continue;
    for (int dk = -r; dk <= r; ++dk)                          // tutti i nodi entro la sferadi raggio MARGINE
        for (int di = -r; di <= r; ++di)
            for (int dj = -r; dj <= r; ++dj) {
                const int kk = k + dk, ii = i + di, jj = j + dj;
                if (kk < 0 || kk >= N || ii < 0 || ii >= N || jj < 0 || jj >= N) continue;
                if ((dk * dk + di * di + dj * dj) * h * h <= MARGINE * MARGINE)
                    margin_mask[(kk * N + ii) * N + jj] = 0;
            }
}
return margin_mask;
}
// Sorgenti della passata 1: i nodi liberi che toccano un ostacolo
inline std::vector<int> nodi_di_bordo(const std::vector<unsigned char>& mask, int N)
{
    std::vector<int> bordo;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
                const int q = (k * N + i) * N + j;
                if (!mask[q]) continue;                                  // solo nodi liberi...
                if ((j > 0 && !mask[q - 1]) || (j < N - 1 && !mask[q + 1]) ||
                    (i > 0 && !mask[q - N]) || (i < N - 1 && !mask[q + N]) ||
                    (k > 0 && !mask[q - N * N]) || (k < N - 1 && !mask[q + N * N]))
                    bordo.push_back(q);                                  // ...che hanno un vicino ostacolo
            }
    return bordo;
}

// FM2: dalla distanza d dagli ostacoli alla lentezza F = 1/v della passata 2
// v = V_MIN + (1 - V_MIN) * min(d / D_SICUREZZA, 1): F = 10 sul bordo, 1 lontano
inline std::vector<float> lentezza_fm2(const std::vector<float>& d)
{
    std::vector<float> F(d.size());
    for (size_t q = 0; q < d.size(); ++q) {
        const double r = std::min((double)d[q] / D_SICUREZZA, 1.0);       // distanza normalizzata: 0 sul bordo, 1 oltre D_SICUREZZA (o se d = INF)
        F[q] = (float)(1.0 / (V_MIN + (1.0 - V_MIN) * r));
    }
    return F;
}
// Controllo iniziale: A e B non devono stare dentro un ostacolo
inline bool ambiente_valido()
{
    return !ostacolo(A.x, A.y, A.z) && !ostacolo(B.x, B.y, B.z);
}

// Dal punto (x, y, z) all'indice del nodo piu' vicino (serve per A e B)
int nodo(int N, double x, double y, double z){
    const double h = 2.0 / (N - 1);
    const int j = (int)lround((x + 1.0) / h), i = (int)lround((y + 1.0) / h), k = (int)lround((z + 1.0) / h);
    return (k * N + i) * N + j;
}

In [ ]:
%%writefile fmm3d.h
#pragma once           // il file viene incluso una sola volta
#include <cstdio>    
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
#include <chrono>      // misura dei tempi
#include <fstream>     // scrittura su file
const float INF = 1e30f;                 // "infinito": nodo non ancora raggiunto dall'onda
typedef std::pair<float, int> Entry;     // coppia (u, indice del nodo) per la coda di priorita' del FMM

// Risolutore locale 3D (Godunov, Alg. 2 esteso al 3D): dai minimi dei vicini a, b, c al nuovo u
// Prova con 1, 2, poi 3 vicini; usato sia dal FMM (CPU) sia dal Block FIM (GPU)
__host__ __device__ float solve_quadratic3(float a, float b, float c,float F, float h)
{
    float Fh = F * h;                              // tempo per attraversare una cella
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}

// Nuovo u del nodo (k, i, j) per il FMM: minimo dei vicini per asse + risolutore locale
float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int k, int i, int j, int N, float h)
{
    int idx = (k * N + i) * N + j;  // indice globale del nodo (k, i, j)
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[idx - 1] : INF;
    float right = (j < N - 1) ? u[idx + 1] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[idx - N] : INF;
    float up    = (i < N - 1) ? u[idx + N] : INF;
    // c: min dei vicini lungo z (stessa colonna e riga, piano k-1 e k+1)
    float back  = (k > 0)     ? u[(idx - N * N)] : INF;
    float front = (k < N - 1) ? u[(idx + N *N)] : INF;

    float a = fminf(left, right);
    float b = fminf(down, up);
    float c = fminf(back, front);

    float q = solve_quadratic3(a, b, c, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ijk, u)
}
// Fast Marching Method (Alg. 3) su CPU: il riferimento per il Block FIM
// Fissa i nodi in ordine di tempo crescente con una coda di priorita'
void fmm(std::vector<float>& u, const std::vector<float>& F, int N, float h, const std::vector<int>& srcs, const std::vector<unsigned char>& mask)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };      // stato di ogni nodo
    std::vector<char> state(N * N * N, LONTANO);    // array di char NxNxN, inizializzato a LONTANO

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    for (int s : srcs){heap.push(Entry(u[s], s)); state[s] = ATTIVO;}

    const int dk[6] = {  0, 0,  0, 0, -1, 1 };       // spostamenti verso i 6 vicini:
    const int di[6] = {  0, 0, -1, 1,  0, 0 };       // sinistra, destra, sotto, sopra,
    const int dj[6] = { -1, 1,  0, 0,  0, 0 };       // davanti, dietro

    while (!heap.empty()) {                          // while A != vuoto
        Entry trial = heap.top();                    // Trial <- radice del min-heap
        heap.pop();                                  // Rimuovi Trial da A
        int x = trial.second;                        // indice del nodo Trial
        if (state[x] == NOTO || trial.first > u[x])  // coppia "scaduta": si ignora
            continue;
        state[x] = NOTO;                             // Aggiungi Trial a Set

        int k = x / (N*N), i = (x / N) % N, j = x % N;                    // strato, riga, colonna del nodo Trial
        for (int d = 0; d < 6; ++d) {                // foreach x_nb adiacente a Trial
            int ni = i + di[d], nj = j + dj[d], nk = k + dk[d];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N || nk < 0 || nk >= N) continue;   // fuori dal dominio
            int nb = (nk * N + ni) * N + nj;   // indice del nodo vicino
            if (state[nb] == NOTO || !mask[nb]) continue;         // if x_nb non appartiene a Set

            float q = solve_node(u, F, nk, ni, nj, N, h);
            if (q < u[nb]) {                         // u(x_nb) <- soluzione di g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                  // Aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}

In [ ]:
%%writefile block_fim3d.cuh
#pragma once           // il file viene incluso una sola volta
// Block FIM 3D (Alg. 7) sulla GPU: blocchi 8^3 in shared memory, uscita anticipata, maschera degli ostacoli
#include <cstdio>
#include <cstdlib>
#include <vector>
#include <algorithm>       // std::min
#include <cuda_runtime.h>
#include "fmm3d.h"         // INF e solve_quadratic3 (lo stesso risolutore locale del FMM)

// Controllo degli errori CUDA: se una chiamata fallisce, stampa dove e perche' ed esce
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

#define BLOCK 8  // blocco 8 x 8 x 8 = 512 nodi, un thread per nodo
// Risolutore locale in shared memory: i 6 vicini del nodo sono nella copia s (blocco + halo)
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}

// Step 1: niter iterazioni; Step 2: 1 iterazione di prova.
__device__ void process_block(int block, int niter, float* u, const float* F,
                             int N, int nbx, float h, const unsigned char* mask, float eps, unsigned char* Cb_out)
{
    // Allocazione statica di shared memory
    __shared__ float s_u[BLOCK + 2][BLOCK + 2][BLOCK + 2];              
    __shared__ bool s_Cn[BLOCK * BLOCK * BLOCK];                  // i flag Cn dei 512 nodi
    // posizione del thread nel suo blocco CUDA: ognuno da 0 a 7
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    // bj: asse x , bi: asse y e bk: asse z; identifica blocco nella griglia
    int bj = block % nbx, bi = (block / nbx) % nbx, bk = block / (nbx * nbx); 
    // numero del thread dentro il blocco; 
    int tid = (tz*BLOCK + ty) * BLOCK + tx; // 0, ..., 511
    // posizione globale rispetto x,y e z: lungo tutta la griglia
    int i = bi * BLOCK + ty, j = bj * BLOCK + tx, k = bk * BLOCK + tz;
    int idx = (k * N + i) * N + j;  // 0, ..., N^3-1
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || !mask[idx];             // nodo da non aggiornare: fuori dalla griglia o maschera
    float f = inside ? F[idx] : 1.0f;
    // Global -> Shared
    for (int m = tid; m < (BLOCK + 2) * (BLOCK + 2) * (BLOCK + 2); m += BLOCK * BLOCK * BLOCK) {
        // posizione nella shared memory
        int sz = m / ((BLOCK + 2) * (BLOCK + 2)), sy = (m / (BLOCK + 2)) % (BLOCK + 2), sx = m % (BLOCK + 2);
        // coordinate globali della cella (sz, sy, sx): -1 perché c'è l'halo                          
        int gi = bi * BLOCK + sy - 1, gj = bj * BLOCK + sx - 1, gk = bk * BLOCK + sz - 1;
        
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();
    bool conv = true;
    for (int it = 0; it < niter; ++it){
        // p: valore del nodo all'inizio di questa iterazione
        float p = s_u[tz+1][ty+1][tx+1];
        // q: nuovo valore calcolato in questa iterazione
        float q;
        if (fixed) {q = p;}
        else{
        // tx, ty, tz: 0,...,7; +1: nella shared memory c'è l'halo
        q = solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);
        }    
        __syncthreads();
        s_u[tz+1][ty + 1][tx + 1] = q;
        conv = fabsf(p - q) < eps;
        if (!__syncthreads_or(!conv)) break;   // tutti hanno scritto; se nessun nodo e' cambiato, esci
    }
    // Shared -> Global: ogni thread riscrive il proprio nodo
    if (inside) u[idx] = s_u[tz+1][ty + 1][tx + 1];
    // Cn del nodo: vero se non è cambiato nell'ultima iterazione
    s_Cn[tid] = conv;
    __syncthreads();
    for (int reduct = BLOCK * BLOCK * BLOCK / 2; reduct > 0; reduct /= 2) {     // 256, ..., 4, 2, 1
        if (tid < reduct) s_Cn[tid] = s_Cn[tid] && s_Cn[tid + reduct];
        __syncthreads();
    }
    if (tid == 0) Cb_out[block] = s_Cn[0];
}
// Step1: ogni blocco della lista A fa niter iterazioni
__global__ void update_active_blocks(const int* active_list, float* u, const float* F,
                                    int N, int nbx, float h, const unsigned char* mask, int niter,
                                    float eps, unsigned char* Cb_step1)
{
    // blockIdx è il numero del blocco CUDA nel lancio in corso
    process_block(active_list[blockIdx.x], niter, u, F, N, nbx, h, mask, eps, Cb_step1);
    // (es. N = 33) giro: 1 -> blocchi CUDA lanciati: 1 -> blockIdx.x: 0 -> block = active_list[blockIdx.x]: 62
    // (es. N = 33) giro: 3 -> blocchi CUDA lanciati: 6 -> blockIdx.x: 0, 1, ..., 5 -> block = active_list[blockIdx.x]: 61, 63, 57, 67, 37, 87
}

// costruisce la lista A dei blocchi attivi (Cb = 0)
__global__ void build_active_list(const unsigned char* Cb, unsigned char* in_list,
                                  int* active_list, int* n_active, int n_blocks) 
{
    // blockIdx: blocco nel lancio in corso; blockDim (monodimensionale CUDA): 256; threadIdx:0,...,255
    int b = blockIdx.x * blockDim.x + threadIdx.x;           // numero del thread(blocco) su tutto il lancio
    // N = 65 -> nbx = 9 -> n_blocks = 729 -> list_blocks = 3 -> thread lanciati = 768 -> thread in eccesso = 39 
    if (b >= n_blocks) return;                               // thread(blocco) in eccesso lanciati da CUDA                                   
    in_list[b] = !Cb[b];                                      // fotografia della lista A di questo giro
    if (!Cb[b]) active_list[atomicAdd(n_active, 1)] = b;               // se NON convergente aggiungi thread(blocco) alla lista A
}
// Step 2: i 6 vicini dei blocchi appena convergenti diventano candidati
__global__ void build_candidate_list(const int* active_list, int n_active, int nbx, const unsigned char* in_list,
                                     const unsigned char* Cb_step1, unsigned char* Cb,
                                     int* is_candidate, int* cand_list, int* cand_count)
{
    int pos = blockIdx.x * blockDim.x + threadIdx.x;
    // se è maggiore o uguale al numero di blocchi attivi fai return
    if (pos >= n_active) return;                       
    int b = active_list[pos];         // un blocco della lista A; N = 33 -> A ={61, 63, ...,87}; active_list[0] =61
    Cb[b] = Cb_step1[b];             // leggi il valore di Cb
    if (!Cb_step1[b]) return;        // proseguono solo i blocchi appena convergenti
    // bj: blocco lungo x; bi: blocco lungo y; bk: blocco lungo z
    int bj = b % nbx, bi = (b / nbx) % nbx, bk = b / (nbx * nbx);
    // sinistra, destra, sotto, sopra e davanti, dietro
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};
    // verifico se si trova nella griglia
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // scarta blocchi fuori dalla griglia
        // scarta blocchi attivi e non convergenti
        if (in_list[nb[d]] && !Cb_step1[nb[d]]) continue;          // d: indica la direzione; N = 33; nb[d] = [61, 63, 57, 67, 37, 87];      
        if (atomicExch(&is_candidate[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}
// Step 2: ogni candidato fa 1 iterazione; se cambia, diventa attivo
__global__ void update_candidate_blocks(const int* cand_list, const int* cand_count, int* is_candidate,
                                        float* u, const float* F, int N, int nbx, float h,
                                        const unsigned char* mask, float eps, unsigned char* Cb)
{
    // scarta blocchi CUDA in eccesso
    if (blockIdx.x >= *cand_count) return;
    int b = cand_list[blockIdx.x];
    // blocchi candidati: niter = 1 -> con la nuova informazione dal blocco convergente, questo vicino cambia?
    process_block(b, 1, u, F, N, nbx, h, mask, eps, Cb);
    if (threadIdx.x == 0 && threadIdx.y == 0 && threadIdx.z == 0) is_candidate[b] = 0;   // pronto per il giro successivo
}
// Ogni giro: lista A -> Step 1 -> Step 2; fine quando A e' vuota
int block_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
              const std::vector<int>& srcs, const std::vector<unsigned char>& mask, float eps, int niter, float* ms)
{
    // nbx: numero di blocchi per lato; n_blocks: blocchi totali
    const int nbx = (N + BLOCK - 1) / BLOCK, n_blocks = nbx * nbx * nbx;
    // bytes: dimensione totale griglia 3D
    const size_t bytes = (size_t)N * N * N * sizeof(float);
    // Cb: si inizializza vettore di n_blocks valori a 1
    std::vector<unsigned char> Cb(n_blocks, 1);
    // strato, riga e colonna del nodo sorgente
    for (int s : srcs){
    const int sk = s / (N * N), si = (s / N) % N, sj = s % N;
    // Cb: vale 1 tranne nel blocco della sorgente (0) 
    Cb[((sk / BLOCK) * nbx + si / BLOCK) * nbx + sj / BLOCK] = 0;
    }

    // Memoria sulla GPU e copia dei dati iniziali (CPU -> GPU)
    float *d_u, *d_F; unsigned char *d_Cb, *d_Cb_step1, *d_in_list, *d_mask; int *d_active_list, *d_n_active;
    int *d_is_candidate, *d_cand_list, *d_cand_count;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_Cb_step1, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_in_list, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_active_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_n_active, sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_mask, N*N*N));
    // Memoria per i blocchi candidati
    CUDA_CHECK(cudaMalloc(&d_is_candidate, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));
    // si copiano d_u, d_F e d_Cb dalla CPU alla GPU
    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb.data(), n_blocks, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_mask, mask.data(), N*N*N, cudaMemcpyHostToDevice));
    // si azzerano d_Cb_step1 e d_is_candidate
    CUDA_CHECK(cudaMemset(d_Cb_step1, 0, n_blocks));
    CUDA_CHECK(cudaMemset(d_is_candidate, 0, n_blocks * sizeof(int)));
    // list_blocks: blocchi CUDA da 256 threads per avere un thread per ogni blocco di nodi 
    const int threads = 256, list_blocks = (n_blocks + threads - 1) / threads;
    // si misura l'ora prima del ciclo
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                     
    int iter = 0;
    while (true) {
        // azzera il numero di blocchi attivi                                           
        int n_active = 0;  // contatore su CPU
        CUDA_CHECK(cudaMemset(d_n_active, 0, sizeof(int)));    // contatore su GPU
        // lancia list_blocks CUDA da 256 threads ciascuno, sufficiente per avere un thread ogni blocco
        build_active_list<<<list_blocks, threads>>>(d_Cb, d_in_list, d_active_list, d_n_active, n_blocks);
        CUDA_CHECK(cudaGetLastError());
        // aggiorna n_active con d_n_active
        CUDA_CHECK(cudaMemcpy(&n_active, d_n_active, sizeof(int), cudaMemcpyDeviceToHost));
        if (n_active == 0) break;  // se n_active == 0, lista A vuota, termina ciclo
        ++iter;                 // conta i giri
        // Step 1: foreach b in A in parallelo; lancia n_active blocchi 8x8x8
        update_active_blocks<<<n_active, dim3(BLOCK, BLOCK, BLOCK)>>>(d_active_list, d_u, d_F, N, nbx, h, d_mask, niter, eps, d_Cb_step1);    // si lanciano n_active blocchi CUDA, ognuno con 512 thread
        CUDA_CHECK(cudaGetLastError());
        // azzeriamo il contatore dei candidati sulla GPU
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));
        // lancia n_active blocchi CUDA da 256 threads
        build_candidate_list<<<(n_active + threads-1) / threads, threads>>>(d_active_list, n_active, nbx, d_in_list, d_Cb_step1, d_Cb,
                                                                          d_is_candidate, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());
        // ogni blocco ha al più 6 vicini, quindi 6*n_active
        update_candidate_blocks<<<std::min(6 * n_active, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, d_is_candidate,
                                                                                    d_u, d_F, N, nbx, h, d_mask, eps, d_Cb);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    // Risultato sulla CPU (GPU -> CPU) e liberazione della memoria
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));CUDA_CHECK(cudaFree(d_Cb));
    CUDA_CHECK(cudaFree(d_Cb_step1)); CUDA_CHECK(cudaFree(d_mask));
    CUDA_CHECK(cudaFree(d_in_list)); CUDA_CHECK(cudaFree(d_active_list)); CUDA_CHECK(cudaFree(d_n_active));
    CUDA_CHECK(cudaFree(d_is_candidate)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    return iter;
}


In [ ]:
%%writefile main.cu
// FM2 (passata 1 + passata 2) con il FMM e con il Block FIM: tabella al variare di N e ripianificazione nel labirinto
#include <cstdio>
#include <algorithm>       // std::min
#include <chrono>          // tempi sulla CPU
#include <fstream>         // salvataggio su file
#include <string>          // nomi dei file delle mappe
#include <cuda_runtime.h>  // prima di fmm3d.h: definisce __host__ __device__
#include "fmm3d.h"         // Fast Marching (CPU)
#include "ambiente.h"      // stanza, ostacoli, maschera, lentezza del FM2
#include "block_fim3d.cuh" // Block FIM (GPU)

// FM2 completo (passata 1 + passata 2) sulla mappa attuale, con il FMM (CPU) o con il Block FIM (GPU).
// Restituisce u (tempo di arrivo da B), la F del FM2 e il tempo di calcolo delle due passate [ms].
void fm2(bool con_block_fim, int Nk, const std::vector<unsigned char>& mask, float eps, int niter,
         std::vector<float>& u, std::vector<float>& Fk, double& t_ms, int& giri)
{
    const float hk = 2.0f / (Nk - 1);
    const int n = Nk * Nk * Nk;
    // prima passata: numero di nodi liberi che toccano un ostacolo
    const std::vector<int> bordo = nodi_di_bordo(mask, Nk);
    // seconda passata: percorso minimo per B          
    const std::vector<int> srcs = { nodo(Nk, B.x, B.y, B.z) };          
    std::vector<float> d(n, INF);
    for (int s : bordo) d[s] = 0.0f;
    const std::vector<float> uno(n, 1.0f);
    u.assign(n, INF);
    for (int s : srcs) u[s] = 0.0f;
    giri = 0;
    if (con_block_fim) {                                     // GPU: tempo del solo ciclo dei giri (come nelle tabelle)
        float ms1 = 0.0f, ms2 = 0.0f;
        // calcolo la distanza dei nodi liberi dagli ostacoli
        block_fim(d, uno, Nk, hk, bordo, mask, eps, niter, &ms1);
        // definisco la lentezza
        Fk = lentezza_fm2(d);
        // tempo di arrivo in B
        giri = block_fim(u, Fk, Nk, hk, srcs, mask, eps, niter, &ms2);
        // tempo impiegato dal FM2 con Block FIM
        t_ms = ms1 + ms2;
    } else {                                                 // CPU: le due chiamate al FMM
        auto t0 = std::chrono::high_resolution_clock::now();
        fmm(d, uno, Nk, hk, bordo, mask);
        Fk = lentezza_fm2(d);
        fmm(u, Fk, Nk, hk, srcs, mask);
        auto t1 = std::chrono::high_resolution_clock::now();
        t_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();
    }
}

// Scrive i file di una mappa per Python: u, maschera, F e ambiente (ostacoli e fori della mappa attuale)
void scrivi_mappa(int Nk, int liv, const std::vector<float>& u, const std::vector<unsigned char>& mask, const std::vector<float>& Fk)
{
    const std::string t = std::to_string(Nk) + "_livello" + std::to_string(liv + 1);
    std::ofstream out("fim_" + t + ".bin", std::ios::binary);
    out.write(reinterpret_cast<const char*>(u.data()), u.size() * sizeof(float));
    std::ofstream om("mask_" + t + ".bin", std::ios::binary);
    om.write(reinterpret_cast<const char*>(mask.data()), mask.size());
    std::ofstream of("F_" + t + ".bin", std::ios::binary);
    of.write(reinterpret_cast<const char*>(Fk.data()), Fk.size() * sizeof(float));
    std::ofstream amb("ambiente_" + t + ".txt");
    amb.precision(15);
    amb << "MARGINE " << MARGINE << "\n";
    amb << "A " << A.x << " " << A.y << " " << A.z << "\n";
    amb << "B " << B.x << " " << B.y << " " << B.z << "\n";
    for (const auto& os : OSTACOLI) amb << "ZONA " << os.x0 << " " << os.x1 << " " << os.y0 << " " << os.y1 << " " << os.z0 << " " << os.z1 << "\n";
    for (const auto& f : FORI)    amb << "FORO " << f.x0 << " " << f.x1 << " " << f.y0 << " " << f.y1 << " " << f.z0 << " " << f.z1 << "\n";
}

int main()
{
    // Informazioni sulla GPU usata
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d | max thread per blocco: %d | shared memory per blocco: %zu KB\n",
           prop.name, prop.multiProcessorCount, prop.maxThreadsPerBlock, prop.sharedMemPerBlock / 1024);
    CUDA_CHECK(cudaFree(0));                                 // inizializza la GPU prima delle misure
    const float eps = 1e-6f;                                 // tolleranza di convergenza
    const int niter = 8;                                     // iterazioni locali massime (uscita anticipata): la migliore con gli ostacoli (prove 12 / 24)

    struct Prova { int N, S; };                              // griglia e numero di mappe per livello
    const std::vector<Prova> PROVE = { { 129, 2 }, { 193, 2 }, { 257, 1 } };   // prova pilota (poi: 129 x 30, 193 x 10, 257 x 5)

    for (const auto& p : PROVE) {
        const int Nk = p.N;
        std::ofstream csv("risultati_" + std::to_string(Nk) + ".csv");    // una riga per mappa
        csv << "livello,seed,K,M,nodi_liberi,giri,t_fmm,t_bfim,speedup,max_diff,uA,dist_AB,tortuosita,raggiunto\n";
        printf("\nN = %d\n  liv  seed   giri   max|FMM-BFIM|   u(A)     t FMM [ms]   t BFIM [ms]   speedup\n", Nk);
        for (int l = 0; l < (int)LIVELLI.size(); ++l) {
            const Livello& L = LIVELLI[l];
            for (int s = 0; s < p.S; ++s) {
                const unsigned seed = 1000 * l + s;          // seed diverso per ogni mappa, uguale tra le griglie
                genera_mappa(L.K, L.M, L.lato_min, L.lato_max, seed);
                if (!ambiente_valido()) { printf("  L%d seed %u: A o B dentro un ostacolo, saltata\n", l + 1, seed); continue; }
                const std::vector<unsigned char> mask = costruisci_maschera(Nk);
                int liberi = 0;
                for (unsigned char c : mask) liberi += c;
                const int iA = nodo(Nk, A.x, A.y, A.z);

                std::vector<float> u_fmm, u_bfim, F_fmm, F_bfim;
                double t_fmm = 0.0, t_bfim = 1e30, t;
                int giri = 0, g;
                fm2(false, Nk, mask, eps, niter, u_fmm, F_fmm, t_fmm, g);       // FMM: 1 misura (dura secondi)
                for (int r = 0; r < 3; ++r) {                                    // Block FIM: il minimo di 3 misure
                    fm2(true, Nk, mask, eps, niter, u_bfim, F_bfim, t, giri);
                    t_bfim = std::min(t_bfim, t);
                }
                float diff = 0.0f;                                               // le due u devono coincidere
                for (size_t q = 0; q < u_fmm.size(); ++q) diff = fmaxf(diff, fabsf(u_fmm[q] - u_bfim[q]));

                const bool raggiunto = u_bfim[iA] < INF;                         // l'onda partita da B arriva in A?
                const double dist = std::sqrt((B.x - A.x) * (B.x - A.x) + (B.y - A.y) * (B.y - A.y) + (B.z - A.z) * (B.z - A.z));
                const double tort = raggiunto ? u_bfim[iA] / dist : 0.0;        // tempo di volo / tempo in linea retta a velocita' piena
                csv << l + 1 << "," << seed << "," << L.K << "," << L.M << "," << liberi << "," << giri << ","
                    << t_fmm << "," << t_bfim << "," << t_fmm / t_bfim << "," << diff << ","
                    << (raggiunto ? u_bfim[iA] : -1.0f) << "," << dist << "," << tort << "," << raggiunto << "\n";
                printf("  L%d  %5u   %4d   %13.2e   %7.4f   %10.1f   %11.2f   %7.1f\n",
                       l + 1, seed, giri, diff, raggiunto ? u_bfim[iA] : -1.0f, t_fmm, t_bfim, t_fmm / t_bfim);
                if (Nk == 129 && s == 0) scrivi_mappa(Nk, l, u_bfim, mask, F_bfim);   // una mappa per livello, per la vista 3D
            }
        }
    }
    return 0;
}


In [ ]:
!nvcc -O3 -std=c++17 -arch=sm_75 main.cu -o fim3d && ./fim3d

In [ ]:
# GARA SU UNA MAPPA CASUALE (usa i file scritti da main.cu a N = 129): scegli il livello e riesegui la cella.
# Due viste affiancate (sinistra Block FIM, destra FMM): i due droni partono insieme da A. Ognuno aspetta in A il tempo di calcolo del FM2 con il suo metodo (misurato da main.cu,
# prima mappa del livello), poi vola lungo il percorso. Le due u coincidono (max|FMM - BFIM| ~ 1e-6): il percorso e' lo stesso,
# cambia solo quando si parte.
import csv
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.interpolate import RegularGridInterpolator

# 0) PARAMETRI
LIVELLO = 3                                                         # 1 (facile) ... 5 (difficile)
N = 129                                                             # griglia dei file scritti da main.cu
METRI_PER_UNITA = 10.0                                              # il cubo [-1, 1]^3 e' una stanza di 20 m di lato
V_MAX = 2.0                                                         # velocita' massima del drone [m/s], dove F = 1
SEC_PER_U = METRI_PER_UNITA / V_MAX                                 # 1 unita' di u = 5 s di volo
MS_FRAME = 60                                                       # il Play mostra un fotogramma ogni 60 ms
BLU, ARANCIO = "#2a78d6", "#eb6834"
h = 2.0 / (N - 1)
x = -1.0 + h * np.arange(N)

# 1) DATI: u da B, maschera, lentezza F, ostacoli della prima mappa del livello e tempi di calcolo dal CSV
def leggi_ambiente(nome):
    amb = {"ZONA": [], "FORO": []}
    for riga in open(nome):
        k, *v = riga.split(); v = [float(t) for t in v]
        if k in ("A", "B"): amb[k] = np.array(v)
        elif k == "MARGINE": amb[k] = v[0]
        else: amb[k].append(tuple(v))
    return amb
t = f"{N}_livello{LIVELLO}"
u = np.fromfile(f"fim_{t}.bin", dtype=np.float32).reshape(N, N, N).astype(np.float64)     # u[k, i, j] = u(z, y, x)
libero = np.fromfile(f"mask_{t}.bin", dtype=np.uint8).reshape(N, N, N) == 1
F = np.fromfile(f"F_{t}.bin", dtype=np.float32).reshape(N, N, N).astype(np.float64)
amb = leggi_ambiente(f"ambiente_{t}.txt")
A, B = amb["A"], amb["B"]
F_i = RegularGridInterpolator((x, x, x), F)
peso = RegularGridInterpolator((x, x, x), libero.astype(float))
with open(f"risultati_{N}.csv") as f:                               # la riga della prima mappa del livello (seed = 1000 * (LIVELLO - 1))
    r = [r for r in csv.DictReader(f) if int(r["livello"]) == LIVELLO and int(r["seed"]) == 1000 * (LIVELLO - 1)][0]
attesa = {"Block FIM": float(r["t_bfim"]) / 1000.0, "FMM": float(r["t_fmm"]) / 1000.0}    # tempo di calcolo del FM2 [s]

# 2) PERCORSO da A: discesa lungo u (26 vicini) + accorciamento (come nel labirinto)
mosse = [(a, b, c) for a in (-1, 0, 1) for b in (-1, 0, 1) for c in (-1, 0, 1) if (a, b, c) != (0, 0, 0)]
nodo = lambda p: tuple(np.round((np.asarray(p)[::-1] + 1.0) / h).astype(int))
coord = lambda n: -1.0 + h * np.array(n[::-1], dtype=float)
def tempo(p, q):                                                    # tempo (unita' di u) lungo il segmento p-q: somma di F * ds
    s = np.linspace(0.0, 1.0, int(np.linalg.norm(q - p) / (h / 4)) + 2)[:, None]
    return np.sum(F_i((p + (s[:-1] + s[1:]) / 2 * (q - p))[:, ::-1])) * np.linalg.norm(q - p) / (len(s) - 1)
def si_vede(p, q):                                                  # il segmento p-q non attraversa ostacoli
    s = np.linspace(0.0, 1.0, int(np.linalg.norm(q - p) / (h / 4)) + 2)[:, None]
    return np.all(peso((p + s * (q - p))[:, ::-1]) > 0.999)
def percorso(p0):
    n, nB = nodo(p0), nodo(B)
    if not libero[n] or u[n] >= 1e29:
        raise SystemExit(f"A = {np.round(p0, 2)} e' dentro un ostacolo o non raggiunge B: livello {LIVELLO} non risolto a N = {N}")
    nodi = [n]
    while n != nB:
        vic = [(n[0] + a, n[1] + b, n[2] + c) for a, b, c in mosse]
        vic = [w for w in vic if all(0 <= s < N for s in w) and libero[w]]
        n = min(vic, key=lambda w: u[w] + 0.5 * (F[n] + F[w]) * h * np.linalg.norm(np.subtract(w, n)))
        nodi.append(n)
    P = [np.asarray(p0, float)] + [coord(w) for w in nodi[1:]]
    tn = np.concatenate([[0], np.cumsum([tempo(P[i], P[i + 1]) for i in range(len(P) - 1)])])
    out, a = [P[0]], 0
    while a < len(P) - 1:
        b = len(P) - 1
        while b > a + 1 and not (si_vede(P[a], P[b]) and tempo(P[a], P[b]) <= tn[b] - tn[a]):
            b -= 1
        out.append(P[b]); a = b
    return np.array(out)
perc = percorso(A)
s = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(perc, axis=0), axis=1))])     # punti fitti del percorso e orario di volo
sf = np.linspace(0, s[-1], max(int(s[-1] / (h / 2)), 2))
Pf = np.stack([np.interp(sf, s, perc[:, c]) for c in range(3)], axis=1)
tf = np.concatenate([[0], np.cumsum(F_i(((Pf[:-1] + Pf[1:]) / 2)[:, ::-1]) * np.diff(sf) * SEC_PER_U)])   # istante [s] in ogni punto, da A
vf = V_MAX / F_i(Pf[:, ::-1])                                       # velocita' [m/s]: v = V_MAX / F
T_volo = tf[-1]
arrivo = {m: attesa[m] + T_volo for m in attesa}
print(f"Livello {LIVELLO} (N = {N}): {r['K']} muri, {r['M']} fori per muro, giri del Block FIM {r['giri']}, u(A) = {float(r['uA']):.3f}, "
      f"tortuosita' {float(r['tortuosita']):.2f}")
for m in attesa:
    print(f"  {m:9s}: calcola il FM2 in {attesa[m] * 1000:8.1f} ms, vola {T_volo:6.1f} s, arriva in B a {arrivo[m]:7.2f} s")
print(f"  il drone del FMM arriva {arrivo['FMM'] - arrivo['Block FIM']:.2f} s dopo (speedup del calcolo {float(r['speedup']):.1f})")

# 3) GEOMETRIA: muri interni con i fori (solo le facce esterne), spigoli della stanza, A e B
interni = [b for b in amb["ZONA"] if b[0] > -0.99 and b[1] < 0.99]
fori_di = lambda b: [f for f in amb["FORO"] if b[0] - 1e-9 <= f[0] and f[1] <= b[1] + 1e-9]
def superficie(b, fori):                                            # facce esterne del muro b con i suoi fori: X, Y, Z, I, J, K
    tagli = [sorted({b[2 * a], b[2 * a + 1]} | {min(max(f[2 * a + s], b[2 * a]), b[2 * a + 1]) for f in fori for s in (0, 1)}) for a in range(3)]
    n = [len(t) - 1 for t in tagli]
    pieno = np.zeros(n, bool)
    for ix in range(n[0]):
        for iy in range(n[1]):
            for iz in range(n[2]):
                c = [(tagli[a][i] + tagli[a][i + 1]) / 2 for a, i in enumerate((ix, iy, iz))]
                pieno[ix, iy, iz] = not any(f[0] <= c[0] <= f[1] and f[2] <= c[1] <= f[3] and f[4] <= c[2] <= f[5] for f in fori)
    X, Y, Z, I, J, K = [], [], [], [], [], []
    for ix, iy, iz in zip(*np.nonzero(pieno)):
        for asse in range(3):
            for verso in (0, 1):
                vic = [ix, iy, iz]; vic[asse] += 1 if verso else -1
                if 0 <= vic[asse] < n[asse] and pieno[tuple(vic)]: continue      # faccia interna: non si disegna
                lo = [tagli[a][i] for a, i in enumerate((ix, iy, iz))]; hi = [tagli[a][i + 1] for a, i in enumerate((ix, iy, iz))]
                q = lo[asse] if verso == 0 else hi[asse]
                u_, w_ = [a for a in range(3) if a != asse]
                o = len(X)
                for du, dw in [(0, 0), (1, 0), (1, 1), (0, 1)]:
                    v = [0.0] * 3; v[asse] = q; v[u_] = (lo[u_], hi[u_])[du]; v[w_] = (lo[w_], hi[w_])[dw]
                    X.append(v[0]); Y.append(v[1]); Z.append(v[2])
                I += [o, o]; J += [o + 1, o + 2]; K += [o + 2, o + 3]
    return X, Y, Z, I, J, K
X, Y, Z, I, J, K = [], [], [], [], [], []
for b in interni:
    Xb, Yb, Zb, Ib, Jb, Kb = superficie(b, fori_di(b)); o = len(X)
    X += Xb; Y += Yb; Z += Zb; I += [o + i for i in Ib]; J += [o + j for j in Jb]; K += [o + k for k in Kb]
muri = go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=K, color="#9a9a9a", opacity=0.35, name=f"{len(interni)} muri con i fori", flatshading=True, hoverinfo="name")
spigoli = []
for a in (-1, 1):
    for b in (-1, 1):
        spigoli += [[(-1, a, b), (1, a, b)], [(a, -1, b), (a, 1, b)], [(a, b, -1), (a, b, 1)]]
stanza = go.Scatter3d(x=sum([[p[0][0], p[1][0], None] for p in spigoli], []), y=sum([[p[0][1], p[1][1], None] for p in spigoli], []),
                      z=sum([[p[0][2], p[1][2], None] for p in spigoli], []), mode="lines", line=dict(color="gray", width=2),
                      name="stanza (20 m)", hoverinfo="skip")
ab = go.Scatter3d(x=[A[0], B[0]], y=[A[1], B[1]], z=[A[2], B[2]], mode="markers+text", text=["  A (partenza)", "  B (arrivo)"],
                  marker=dict(size=8, color=["green", "black"]), textfont=dict(size=14), name="A e B")
linea = go.Scatter3d(x=Pf[:, 0], y=Pf[:, 1], z=Pf[:, 2], mode="lines", line=dict(color="black", width=2, dash="dash"),
                     name=f"percorso FM2 (volo {T_volo:.1f} s)", hoverinfo="skip")

# 4) ANIMAZIONE affiancata: a sinistra il drone del Block FIM, a destra quello del FMM (stessi istanti).
# La scia e' colorata con la velocita' (v = V_MAX / F); il pallino ha il colore del metodo.
def posizione(m, t):                                                # posizione e velocita' del drone del metodo m all'istante t
    tv = t - attesa[m]                                              # tempo da quando e' partito
    if tv <= 0: return A, 0.0
    if tv >= T_volo: return B, 0.0
    k = np.searchsorted(tf, tv, side="right"); a = (tv - tf[k - 1]) / max(tf[k] - tf[k - 1], 1e-12)
    return Pf[k - 1] + a * (Pf[k] - Pf[k - 1]), vf[k - 1] + a * (vf[k] - vf[k - 1])
def stato(m, t):
    if t < attesa[m]: return f"calcola il percorso ({t * 1000:.0f} / {attesa[m] * 1000:.0f} ms)"
    if t >= arrivo[m]: return f"arrivato a {arrivo[m]:.2f} s"
    return f"in volo a {posizione(m, t)[1]:.1f} m/s"
colore = {"Block FIM": BLU, "FMM": ARANCIO}
scena_di = {"Block FIM": "scene", "FMM": "scene2"}
def dati(t):
    out = []
    for m in ("Block FIM", "FMM"):
        pos, vel = posizione(m, t)
        tv = np.clip(t - attesa[m], 0.0, T_volo)
        dentro = tf <= tv
        scia, vs = np.vstack([Pf[dentro], pos[None]]), np.append(vf[dentro], vel if tv > 0 else vf[0])
        sc = scena_di[m]
        out += [go.Scatter3d(x=scia[:, 0], y=scia[:, 1], z=scia[:, 2], mode="lines", scene=sc, showlegend=False, name=f"scia {m}",
                             line=dict(color=vs, colorscale="Turbo", cmin=0.0, cmax=V_MAX, width=12,
                                       colorbar=dict(title="velocita'<br>[m/s]", x=0.47, len=0.6) if m == "Block FIM" else None,
                                       showscale=(m == "Block FIM"))),
                go.Scatter3d(x=[pos[0]], y=[pos[1]], z=[pos[2]], mode="markers", scene=sc, showlegend=False, name=f"drone {m}",
                             marker=dict(size=8, color=colore[m], line=dict(color="white", width=2)))]
    return out
def titolo(t):
    return (f"Livello {LIVELLO}, t = {t:5.2f} s<br><span style='color:{BLU}'>Block FIM: {stato('Block FIM', t)}</span>  |  "
            f"<span style='color:{ARANCIO}'>FMM: {stato('FMM', t)}</span>")
t_fine = max(arrivo.values())
inizio = min(attesa["FMM"] + 1.0, t_fine)                           # fino a 1 s dopo la partenza del FMM: un fotogramma ogni 0.05 s
passo = max(0.5, T_volo / 250)                                      # poi al massimo ~250 fotogrammi (file HTML leggero anche per i voli lunghi)
istanti = np.unique(np.concatenate([np.arange(0.0, inizio, 0.05), np.arange(inizio, t_fine, passo), [attesa["Block FIM"], attesa["FMM"], t_fine]]))
fisse = []
for sc in ("scene", "scene2"):                                      # stanza, muri, A e B, percorso: in tutte e due le viste
    for tr in (stanza, muri, ab, linea):
        tr = go.Figure(tr).data[0]; tr.update(scene=sc, showlegend=(sc == "scene")); fisse.append(tr)
mobili = list(range(len(fisse), len(fisse) + len(dati(0.0))))
fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]], horizontal_spacing=0.02,
                    subplot_titles=(f"Block FIM (GPU): calcola in {attesa['Block FIM'] * 1000:.0f} ms, arriva a {arrivo['Block FIM']:.2f} s",
                                    f"FMM (CPU): calcola in {attesa['FMM'] * 1000:.0f} ms, arriva a {arrivo['FMM']:.2f} s"))
for tr in fisse + dati(0.0): fig.add_trace(tr)
fig.frames = [go.Frame(data=dati(t), traces=mobili, name=f"{t:.2f}", layout=dict(title=dict(text=titolo(t)))) for t in istanti]
ASSI = dict(range=[-1.0, 1.0], autorange=False, showticklabels=False, title="")
SCENA = dict(xaxis=ASSI, yaxis=ASSI, zaxis=ASSI, aspectmode="cube", camera=dict(eye=dict(x=0.3, y=-1.9, z=1.1)))
fig.update_layout(title=dict(text=titolo(0.0), x=0.5, font=dict(size=14)), scene=SCENA, scene2=SCENA, height=760,
                  margin=dict(l=0, r=0, t=90, b=110), legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.02),
                  updatemenus=[dict(type="buttons", direction="left", x=0.08, y=0.0, xanchor="right", yanchor="top", pad=dict(r=10, t=60), buttons=[
                      dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=MS_FRAME, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
                      dict(label="Pausa", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
                  sliders=[dict(x=0.1, len=0.88, y=0.0, yanchor="top", currentvalue=dict(prefix="tempo t = ", suffix=" s"), pad=dict(t=40),
                                steps=[dict(label="", method="animate", args=[[f"{t:.2f}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                                       for t in istanti])])
fig.write_html(f"gara_livello{LIVELLO}.html", include_plotlyjs="cdn")      # anche come file da scaricare: si apre nel browser
fig.show()
